In [55]:
!pip3 install -r requirements.txt
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip3 install matplotlib pandas numpy scikit-learn yfinance pandas-ta tqdm seaborn plotly ipywidgets

Looking in indexes: https://download.pytorch.org/whl/cu118


In [56]:
# Core imports
import os
import random
import warnings

import numpy as np

warnings.filterwarnings('ignore')

# Data visualization
%matplotlib inline

# Machine learning
import torch
import pandas as pd

In [57]:
TINKOFF_API_PROD = 'invest-public-api.tinkoff.ru:443'
TINKOFF_API_SANDBOX = 'sandbox-invest-public-api.tinkoff.ru:443'

In [4]:
INVEST_API_KEY = input("Enter ML developer Invest API Key: ")

In [5]:
# Check CUDA availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
print(f"Current CUDA device: {torch.cuda.current_device()}")
print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.7.0+cu118
CUDA available: True
CUDA device count: 1
Current CUDA device: 0
CUDA device name: NVIDIA GeForce RTX 4080 SUPER


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [7]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()

In [8]:
from TInvestDataProvider import TInvestDataProvider
from datetime import datetime, timedelta
from externalClients.TInvestApi.proto.marketdata_pb2 import (
    CandleInterval
)



In [9]:
import pytz
from datetime import datetime

moscow_tz = pytz.timezone('Europe/Moscow')
naive_datetime = datetime(year=2025, month=4, day=24, hour=11, second=1)
moscow_datetime = moscow_tz.localize(naive_datetime)
train_period_end = moscow_datetime.astimezone(pytz.UTC)

In [13]:
import asyncio

provider = TInvestDataProvider(api_key=INVEST_API_KEY)
TARGET_CANDLE_COUNT = 1_000_000
RATE_LIMITER_DELAY = 0.25   # Can go as low as 0.2
timestep = timedelta(hours=40)

df = None

try:
    current_end = train_period_end
    step_count = 1
    while True:
        print(f"Step {step_count}. ", end="")
        step_count += 1
        current_start = current_end - timestep

        new_df = await provider.get_historical_candles(
            instrument_id="e6123145-9665-43e0-8413-cd61b8aa9b13",
            from_time=current_start,
            to_time=current_end,
            interval=CandleInterval.CANDLE_INTERVAL_1_MIN
        )

        if df is None:
            df = new_df
        else:
            df = pd.concat([new_df, df])

        if len(df) >= TARGET_CANDLE_COUNT:
            df = df.iloc[-TARGET_CANDLE_COUNT:]
            print(f"Current df shape: {df.shape}")
            break

        current_end = current_start
        print(f"Current df shape: {df.shape}; new candles fetched = {new_df.shape[0]}")
        asyncio.sleep(RATE_LIMITER_DELAY)
except Exception as e:
    print(f"Unable to fetch data {e}")
    last_request_sent = {
        "instrument_id": "e6123145-9665-43e0-8413-cd61b8aa9b13",
        "from_time": current_start,
        "to_time": current_end,
        "interval": CandleInterval.CANDLE_INTERVAL_1_MIN
    }
    print(f"Last executed request: {last_request_sent}")
finally:
    await provider.close()


Step 1. Current df shape: (1520, 7); new candles fetched = 1520
Step 2. Current df shape: (3218, 7); new candles fetched = 1698
Step 3. Current df shape: (4966, 7); new candles fetched = 1748
Step 4. Current df shape: (6545, 7); new candles fetched = 1579
Step 5. Current df shape: (8243, 7); new candles fetched = 1698
Step 6. Current df shape: (9986, 7); new candles fetched = 1743
Step 7. Current df shape: (11535, 7); new candles fetched = 1549
Step 8. Current df shape: (13395, 7); new candles fetched = 1860
Step 9. Current df shape: (15138, 7); new candles fetched = 1743
Step 10. Current df shape: (16658, 7); new candles fetched = 1520
Step 11. Current df shape: (18473, 7); new candles fetched = 1815
Step 12. Current df shape: (20485, 7); new candles fetched = 2012
Step 13. Current df shape: (22005, 7); new candles fetched = 1520
Step 14. Current df shape: (23703, 7); new candles fetched = 1698
Step 15. Current df shape: (25459, 7); new candles fetched = 1756
Step 16. Current df shape

In [11]:
df.shape

(80000, 7)

In [14]:
from FeatureEngineer import FeatureEngineer

feature_engineer = FeatureEngineer(lookback_window=60)
augmented_df = feature_engineer.add_features(df)

In [16]:
!pip install -r requirements.txt  --exists-action w

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5


In [17]:
!pip uninstall numpy scipy scikit-learn torch torchvision torchaudio -y

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.15.2
Uninstalling scipy-1.15.2:
  Successfully uninstalled scipy-1.15.2
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: torch 2.7.0+cu118
Uninstalling torch-2.7.0+cu118:
  Successfully uninstalled torch-2.7.0+cu118
Found existing installation: torchvision 0.22.0+cu118
Uninstalling torchvision-0.22.0+cu118:
  Successfully uninstalled torchvision-0.22.0+cu118
Found existing installation: torchaudio 2.7.0+cu118
Uninstalling torchaudio-2.7.0+cu118:
  Successfully uninstalled torchaudio-2.7.0+cu118


You can safely remove it manually.


In [18]:
!pip install numpy==1.26.4 scipy==1.14.1 scikit-learn==1.2.2
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pandas==1.3.5 tqdm==4.64.1

  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached scipy-1.14.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached scikit_learn-1.2.2-cp311-cp311-win_amd64.whl.metadata (11 kB)
Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl (15.8 MB)
Using cached scipy-1.14.1-cp311-cp311-win_amd64.whl (44.8 MB)
Using cached scikit_learn-1.2.2-cp311-cp311-win_amd64.whl (8.3 MB)
Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.0%2Bcu118-cp311-cp311-win_amd64.whl.metadata (29 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.0%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.0%2Bcu118-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
Using cached https://download.pytorch.org/whl/cu118/torch-2.7.0%2Bcu118-cp311-cp311-win_amd64.whl (2908.5 MB)
Using cached https://download.pytorch.org/whl/cu118/torc

  You can safely remove it manually.


In [15]:
from sklearn.utils import compute_class_weight
from sklearn.model_selection import TimeSeriesSplit
import torch.nn as nn

# Ensure proper scaling
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(augmented_df.drop('target_direction', axis=1))
y = augmented_df['target_direction'].values

# Handle class imbalance
class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

In [46]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm import tqdm
import numpy as np

MINUTES_IN_HOUR = 60

# Configuration
class Config:
    SEED = 42
    BATCH_SIZE = 256
    HIDDEN_SIZE = 128
    NUM_LAYERS = 2
    DROPOUT = 0.3
    LEARNING_RATE = 1e-1
    EPOCHS = 100
    PATIENCE = 5
    SEQ_LEN = MINUTES_IN_HOUR // 2


In [47]:
class EnhancedTradingModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,  # Capture both directions
            dropout=0.1 if num_layers > 1 else 0
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_size*2, hidden_size),  # *2 for bidirectional
            nn.GELU(),  # Smoother than Tanh
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, 1),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, hidden_size),
            nn.GELU(),
            nn.LayerNorm(hidden_size),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, 2)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = self.attention(lstm_out)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return self.classifier(context)

In [48]:
class EarlyStopper:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float('inf')

    def __call__(self, validation_loss):
        if validation_loss < self.min_validation_loss - self.min_delta:
            self.min_validation_loss = validation_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

def create_sequences(data, seq_length):
    sequences = []
    targets = []
    for i in range(len(data)-seq_length):
        sequences.append(data[i:i+seq_length, :-1])  # Exclude target
        targets.append(data[i+seq_length, -1])       # Last element is target
    return np.array(sequences), np.array(targets)


In [49]:
def train_test_split(X, y, test_size=0.2, shuffle=True, random_state=None):
    """Custom train-test split implementation without scikit-learn"""
    if random_state:
        np.random.seed(random_state)

    n_samples = len(X)
    test_samples = int(n_samples * test_size)

    if shuffle:
        indices = np.random.permutation(n_samples)
    else:
        indices = np.arange(n_samples)

    test_indices = indices[:test_samples]
    train_indices = indices[test_samples:]

    X_train, X_test = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    return X_train, X_test, y_train, y_test

In [50]:
# Training Loop
def train_model(df):
    # Prepare data
    data = df.values.astype(np.float32)
    X, y = create_sequences(data, Config.SEQ_LEN)

    # Initialize TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)  # Typically 3-5 splits for time series

    # We'll use the last split as our validation set
    for train_index, val_index in tscv.split(X):
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]
        break  # We just want the last split for train/val

    # Convert to tensors
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    # Rest of your training code remains the same...
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)  # Never shuffle time series!
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    input_size = X_train.shape[-1]
    model = EnhancedTradingModel(
        input_size=input_size,
        hidden_size=Config.HIDDEN_SIZE,
        num_layers=Config.NUM_LAYERS
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(),
                           lr=1e-4,
                           weight_decay=1e-5)  # L2 regularization

    # Gradient clipping
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=1e-3,
        steps_per_epoch=len(train_loader),
        epochs=Config.EPOCHS
    )

    best_val_loss = float('inf')
    try:
        for epoch in range(Config.EPOCHS):
            # Training
            model.train()
            train_loss = 0
            progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS} [Train]')
            for inputs, targets in progress_bar:
                inputs, targets = inputs.to(device), targets.long().to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                train_loss += loss.item()
                #progress_bar.set_postfix(loss=loss.item())

            # Validation
            model.eval()
            val_loss = 0
            correct = 0
            total = 0
            with torch.no_grad():
                val_progress = tqdm(val_loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS} [Val]')
                for inputs, targets in val_progress:
                    inputs, targets = inputs.to(device), targets.long().to(device)

                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()

                    _, predicted = torch.max(outputs.data, 1)
                    total += targets.size(0)
                    correct += (predicted == targets).sum().item()

                    #val_progress.set_postfix(loss=loss.item(), acc=100*correct/total)

            # Epoch statistics
            train_loss /= len(train_loader)
            val_loss /= len(val_loader)
            val_acc = 100 * correct / total

            print(f'Epoch {epoch+1} | '
                  f'Train Loss: {train_loss:.4f} | '
                  f'Val Loss: {val_loss:.4f} | '
                  f'Val Acc: {val_acc:.2f}%')

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), '../best_model.pth')
                print(f'New best model saved with val loss {val_loss:.4f}')

            scheduler.step(val_loss)

    except KeyboardInterrupt:
        print("\nTraining interrupted by user. Saving current model...")
        torch.save(model.state_dict(), '../interrupted_model.pth')

    return model


In [51]:
def train_model(df):
    # Prepare data
    data = df.values.astype(np.float32)
    X, y = create_sequences(data, Config.SEQ_LEN)

    # Time-based split
    tscv = TimeSeriesSplit(n_splits=5)
    for train_index, val_index in tscv.split(X):
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]
        break  # Take last split

    # Convert to tensors
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    # Initialize model
    input_size = X_train.shape[-1]
    model = EnhancedTradingModel(
        input_size=input_size,
        hidden_size=Config.HIDDEN_SIZE,
        num_layers=Config.NUM_LAYERS
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=1e-3,
        steps_per_epoch=len(train_loader),
        epochs=Config.EPOCHS
    )

    best_val_loss = float('inf')

    print("\nStarting training...")
    print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

    try:
        for epoch in range(Config.EPOCHS):
            # Training
            model.train()
            train_loss = 0
            for inputs, targets in train_loader:
                inputs, targets = inputs.to(device), targets.long().to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

                train_loss += loss.item()

            # Validation
            model.eval()
            val_loss = 0
            correct = 0
            total = 0
            with torch.no_grad():
                for inputs, targets in val_loader:
                    inputs, targets = inputs.to(device), targets.long().to(device)

                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                    val_loss += loss.item()

                    _, predicted = torch.max(outputs.data, 1)
                    total += targets.size(0)
                    correct += (predicted == targets).sum().item()

            # Epoch statistics
            train_loss /= len(train_loader)
            val_loss /= len(val_loader)
            val_acc = 100 * correct / total

            print(f'\nEpoch {epoch+1}/{Config.EPOCHS}: '
                  f'Train Loss: {train_loss:.4f} '
                  f'Val Loss: {val_loss:.4f} '
                  f'Val Accuracy: {val_acc:.2f}%',
                  end="")

            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), '../best_model.pth')
                print('   [Best model saved]', end='')

            scheduler.step()

    except KeyboardInterrupt:
        print("\nTraining interrupted. Saving current model...")
        torch.save(model.state_dict(), '../interrupted_model.pth')

    print("\nTraining completed!")
    return model

In [52]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [53]:
print("Unique target values:", augmented_df['target_direction'].unique())
print("Target value counts:\n", augmented_df['target_direction'].value_counts())

Unique target values: [0 1]
Target value counts:
 target_direction
0    431677
1    320739
Name: count, dtype: int64


In [54]:
#train_data = augmented_df.iloc[:10000]

model = train_model(augmented_df)


sample_input = torch.randn(1, Config.SEQ_LEN, 25).to(device)  # 25 features
with torch.no_grad():
    output = model(sample_input)
print("Sample output:", torch.softmax(output, dim=1).cpu().numpy())


Starting training...
Training samples: 125401, Validation samples: 125397

Epoch 1/100: Train Loss: 0.6937 Val Loss: 0.6907 Val Accuracy: 54.03%   [Best model saved]
Epoch 2/100: Train Loss: 0.6866 Val Loss: 0.6902 Val Accuracy: 54.12%   [Best model saved]
Epoch 3/100: Train Loss: 0.6860 Val Loss: 0.6903 Val Accuracy: 54.05%
Epoch 4/100: Train Loss: 0.6861 Val Loss: 0.6933 Val Accuracy: 53.46%
Epoch 5/100: Train Loss: 0.6853 Val Loss: 0.6922 Val Accuracy: 53.93%
Epoch 6/100: Train Loss: 0.6847 Val Loss: 0.6923 Val Accuracy: 53.86%
Epoch 7/100: Train Loss: 0.6846 Val Loss: 0.6921 Val Accuracy: 53.91%
Epoch 8/100: Train Loss: 0.6842 Val Loss: 0.6922 Val Accuracy: 53.86%
Epoch 9/100: Train Loss: 0.6843 Val Loss: 0.6918 Val Accuracy: 54.05%
Epoch 10/100: Train Loss: 0.6844 Val Loss: 0.6916 Val Accuracy: 54.13%
Epoch 11/100: Train Loss: 0.6843 Val Loss: 0.6918 Val Accuracy: 54.12%
Epoch 12/100: Train Loss: 0.6841 Val Loss: 0.6917 Val Accuracy: 54.14%
Epoch 13/100: Train Loss: 0.6840 Val Lo

In [61]:

augmented_df['target_direction'].unique()[:50]

array([0, 1])

In [103]:
class TradingTransformer(nn.Module):
    def __init__(self, input_size, num_classes=5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        self.temporal = nn.LSTM(128, 256, batch_first=True)
        self.attention = nn.MultiheadAttention(256, num_heads=4)

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.encoder(x)
        x, _ = self.temporal(x)
        x = x.permute(1, 0, 2)  # (seq, batch, features)
        x, _ = self.attention(x, x, x)
        x = x.mean(dim=0)  # Global average pooling
        return self.classifier(x)

In [104]:
from ml.EnhancedFeatureEngineer import EnhancedFeatureEngineer
from torchmetrics import Accuracy, F1Score


def train_transformer_model(df, num_classes=5):
    X, y = create_sequences(df, Config.SEQ_LEN)
    y = y.astype(np.int64)  # Ensure targets are integers

    # Time-based split (no shuffling)
    split_idx = int(0.8 * len(X))
    X_train, y_train = X[:split_idx], y[:split_idx]
    X_val, y_val = X[split_idx:], y[split_idx:]

    # 2. Create Datasets and Loaders
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))

    train_loader = DataLoader(train_dataset,
                            batch_size=Config.BATCH_SIZE,
                            shuffle=False)  # Critical for time series!

    val_loader = DataLoader(val_dataset,
                          batch_size=Config.BATCH_SIZE)

    # 3. Initialize Model
    input_size = X_train.shape[-1]
    model = TradingTransformer(input_size, num_classes).to(device)

    # 4. Loss and Optimizer
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights).to(device))
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # 5. Metrics
    val_accuracy = Accuracy(task='multiclass', num_classes=num_classes).to(device)
    val_f1 = F1Score(task='multiclass', num_classes=num_classes).to(device)

    # 6. Training Loop
    best_val_loss = float('inf')
    for epoch in range(Config.EPOCHS):
        # Training Phase
        model.train()
        train_loss = 0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS} [Train]')

        for inputs, targets in progress_bar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        # Validation Phase
        model.eval()
        val_loss = 0
        val_accuracy.reset()
        val_f1.reset()

        with torch.no_grad():
            val_progress = tqdm(val_loader, desc=f'Epoch {epoch+1}/{Config.EPOCHS} [Val]')
            for inputs, targets in val_progress:
                inputs, targets = inputs.to(device), targets.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()

                # Update metrics
                preds = torch.argmax(outputs, dim=1)
                val_accuracy.update(preds, targets)
                val_f1.update(preds, targets)

                val_progress.set_postfix({
                    'loss': loss.item(),
                    'acc': val_accuracy.compute().item(),
                    'f1': val_f1.compute().item()
                })

        # Epoch Statistics
        train_loss /= len(train_loader)
        val_loss /= len(val_loader)
        val_acc = val_accuracy.compute().item()
        val_f1_score = val_f1.compute().item()

        print(f'\nEpoch {epoch+1} Results:')
        print(f'Train Loss: {train_loss:.4f}')
        print(f'Val Loss: {val_loss:.4f}')
        print(f'Val Accuracy: {val_acc:.2%}')
        print(f'Val F1: {val_f1_score:.4f}')

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_transformer.pth')
            print('Saved new best model!')

        # Early stopping check
        if epoch > Config.PATIENCE and val_loss > best_val_loss:
            print(f'Early stopping at epoch {epoch+1}')
            break

    return model

In [110]:
feature_engineer = EnhancedFeatureEngineer()
processed_df = feature_engineer.add_features(df)

data = processed_df.values.astype(np.float32)

KeyboardInterrupt: 

In [109]:
X, y = create_sequences(data, Config.SEQ_LEN)
y = y.astype(np.int64)  # Ensure targets are integers

# Time-based split (no shuffling)
split_idx = int(0.8 * len(X))
X_train, y_train = X[:split_idx], y[:split_idx]
X_val, y_val = X[split_idx:], y[split_idx:]

In [107]:
unique_classes = np.unique(y_train)
unique_classes

array([0, 1, 2, 3])

In [79]:
model = train_transformer_model(df)

Epoch 1/100 [Train]:   0%|          | 0/250 [00:00<?, ?it/s]


RuntimeError: weight tensor should be defined either for all 5 classes or no classes but got weight tensor of shape: [4]